In [1]:
import pandas as pd

import mlflow
from mlflow.models import infer_signature
from mlflow.data.pandas_dataset import PandasDataset


from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True, as_frame=True)

# data_md5 = hashlib.md5(json.dumps(data, sort_keys=True).encode('utf-8')).hexdigest()

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
X.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [4]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 200,
    "multi_class": "multinomial",
    "random_state": 74,
}

In [5]:
clf = DecisionTreeClassifier(max_depth=4, criterion='gini')
clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=4)

In [6]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

/opt/miniconda3/envs/otus/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=200, multi_class='multinomial', random_state=74)

In [7]:
# Predict on the test set
y_pred_lr = lr.predict(X_test)
y_pred_clf = clf.predict(X_test)

In [8]:
# Calculate metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)
accuracy_clf = accuracy_score(y_test, y_pred_clf)
f1_score_lr = f1_score(y_test, y_pred_lr, average='weighted')
accuracy_lr, accuracy_clf

(1.0, 1.0)

In [10]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:5005")

# Create a new MLflow Experiment
mlflow.set_experiment("MLflow MLAdv practice")

2025/05/19 20:45:03 INFO mlflow.tracking.fluent: Experiment with name 'MLflow MLAdv practice' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1747676703060, experiment_id='1', last_update_time=1747676703060, lifecycle_stage='active', name='MLflow MLAdv practice', tags={}>

In [44]:
for i in range(3):
    # Start an MLflow run
    with mlflow.start_run(run_name=f'Run {i+1} - some model run'):
        # Log the hyperparameters
        mlflow.log_params(params)

        # Log the loss metric
        mlflow.log_metric("LogisticRegression accuracy", accuracy_lr)
        # mlflow.log_metric("DecisionTreeClassifier accuracy", accuracy_clf)
        mlflow.log_metric('Logistic f1 score', f1_score_lr)

        # Set a tag that we can use to remind ourselves what this run was for
        mlflow.set_tag("Training Info", "Basic LR model for iris data")

        # Infer the model signature
        signature = infer_signature(X_train, lr.predict(X_train))

        # Log the model
        model_info = mlflow.sklearn.log_model(
            sk_model=lr,
            artifact_path="iris_model",
            signature=signature,
            input_example=X_train,
            registered_model_name="tracking-quickstart",
        )

        dataset: PandasDataset = mlflow.data.from_pandas(X_train)
        mlflow.log_input(dataset, context="training")


Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/05/19 21:04:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 4
Created version '4' of model 'tracking-quickstart'.


🏃 View run Run 1 - some model run at: http://127.0.0.1:5005/#/experiments/1/runs/f92c7040064848ecb3a5c3d791059447
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/1


Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/05/19 21:04:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 5
Created version '5' of model 'tracking-quickstart'.


🏃 View run Run 2 - some model run at: http://127.0.0.1:5005/#/experiments/1/runs/9c6cd42318d34733a8cda062aae420a2
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/1


Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2025/05/19 21:04:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 6


🏃 View run Run 3 - some model run at: http://127.0.0.1:5005/#/experiments/1/runs/985ddfffee9745ad9d66d907ba179947
🧪 View experiment at: http://127.0.0.1:5005/#/experiments/1


Created version '6' of model 'tracking-quickstart'.


## Поиск экспериментов на сервере

In [45]:
experiments = mlflow.search_experiments()
# experiments = pd.DataFrame(experiments)
experiments

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1747676703060, experiment_id='1', last_update_time=1747676703060, lifecycle_stage='active', name='MLflow MLAdv practice', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1747674724657, experiment_id='0', last_update_time=1747674724657, lifecycle_stage='active', name='Default', tags={}>]

In [46]:
type(experiments[0])

mlflow.entities.experiment.Experiment

In [47]:
experiment_name = 'MLflow MLAdv practice'

In [48]:
df = mlflow.search_runs(experiment_names=[experiment_name], order_by=["metrics.m DESC"])

In [49]:
df

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.LogisticRegression accuracy,metrics.Logistic f1 score,params.random_state,params.multi_class,params.solver,params.max_iter,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.log-model.history,tags.mlflow.source.type,tags.Training Info,tags.mlflow.runName
0,985ddfffee9745ad9d66d907ba179947,1,FINISHED,mlflow-artifacts:/1/985ddfffee9745ad9d66d907ba...,2025-05-19 18:04:37.194000+00:00,2025-05-19 18:04:39.968000+00:00,1.0,1.0,74,multinomial,lbfgs,200,stureiko,/opt/miniconda3/envs/otus/lib/python3.12/site-...,"[{""run_id"": ""985ddfffee9745ad9d66d907ba179947""...",LOCAL,Basic LR model for iris data,Run 3 - some model run
1,9c6cd42318d34733a8cda062aae420a2,1,FINISHED,mlflow-artifacts:/1/9c6cd42318d34733a8cda062aa...,2025-05-19 18:04:34.638000+00:00,2025-05-19 18:04:37.165000+00:00,1.0,1.0,74,multinomial,lbfgs,200,stureiko,/opt/miniconda3/envs/otus/lib/python3.12/site-...,"[{""run_id"": ""9c6cd42318d34733a8cda062aae420a2""...",LOCAL,Basic LR model for iris data,Run 2 - some model run
2,f92c7040064848ecb3a5c3d791059447,1,FINISHED,mlflow-artifacts:/1/f92c7040064848ecb3a5c3d791...,2025-05-19 18:04:30.908000+00:00,2025-05-19 18:04:34.604000+00:00,1.0,1.0,74,multinomial,lbfgs,200,stureiko,/opt/miniconda3/envs/otus/lib/python3.12/site-...,"[{""run_id"": ""f92c7040064848ecb3a5c3d791059447""...",LOCAL,Basic LR model for iris data,Run 1 - some model run
3,8ef3c6d9057a4255beecabf2f2ece96c,1,FINISHED,mlflow-artifacts:/1/8ef3c6d9057a4255beecabf2f2...,2025-05-19 17:59:49.019000+00:00,2025-05-19 17:59:51.858000+00:00,1.0,1.0,74,multinomial,lbfgs,200,stureiko,/opt/miniconda3/envs/otus/lib/python3.12/site-...,"[{""run_id"": ""8ef3c6d9057a4255beecabf2f2ece96c""...",LOCAL,Basic LR model for iris data,Run 3 - another basic model
4,196812af29b74b89adb85db565005f0e,1,FINISHED,mlflow-artifacts:/1/196812af29b74b89adb85db565...,2025-05-19 17:59:30.382000+00:00,2025-05-19 17:59:34.470000+00:00,1.0,1.0,74,multinomial,lbfgs,200,stureiko,/opt/miniconda3/envs/otus/lib/python3.12/site-...,"[{""run_id"": ""196812af29b74b89adb85db565005f0e""...",LOCAL,Basic LR model for iris data,Run 1 - baseline model
5,d2b5f64990024dcb87df3f4f52922b9a,1,FINISHED,mlflow-artifacts:/1/d2b5f64990024dcb87df3f4f52...,2025-05-19 17:45:29.878000+00:00,2025-05-19 17:45:37.132000+00:00,1.0,1.0,74,multinomial,lbfgs,200,stureiko,/opt/miniconda3/envs/otus/lib/python3.12/site-...,"[{""run_id"": ""d2b5f64990024dcb87df3f4f52922b9a""...",LOCAL,Basic LR model for iris data,Run 2 - next parameters


In [50]:
import mlflow

# Получение клиента
client = mlflow.MlflowClient()

# Получение всех экспериментов
experiments = client.search_experiments()

# Выводим список экспериментов
for exp in experiments:
    print(f"Experiment ID: {exp.experiment_id}, Name: {exp.name}, Artifact Location: {exp.artifact_location}")

Experiment ID: 1, Name: MLflow MLAdv practice, Artifact Location: mlflow-artifacts:/1
Experiment ID: 0, Name: Default, Artifact Location: mlflow-artifacts:/0


In [51]:
# Для каждого эксперимента получаем запуски
for exp in experiments:
    print(f"\n=== Runs for Experiment: {exp.name} (ID: {exp.experiment_id}) ===")

    # Получаем запуски в эксперименте (можно задать фильтры)
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="",  # можно фильтровать, например "metrics.LogisticRegression_accuracy > 0.9"
        run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
    )

    for run in runs:
        print(f"- Run ID: {run.info.run_id}")
        print(f"  Status: {run.info.status}")
        print(f"  Start Time: {run.info.start_time}")
        print(f"  Metrics: {run.data.metrics}")
        print(f"  Params: {run.data.params}")
        print(f"  Tags: {run.data.tags}")


=== Runs for Experiment: MLflow MLAdv practice (ID: 1) ===
- Run ID: 985ddfffee9745ad9d66d907ba179947
  Status: FINISHED
  Start Time: 1747677877194
  Metrics: {'LogisticRegression accuracy': 1.0, 'Logistic f1 score': 1.0}
  Params: {'solver': 'lbfgs', 'max_iter': '200', 'multi_class': 'multinomial', 'random_state': '74'}
  Tags: {'mlflow.user': 'stureiko', 'mlflow.source.name': '/opt/miniconda3/envs/otus/lib/python3.12/site-packages/ipykernel_launcher.py', 'mlflow.source.type': 'LOCAL', 'mlflow.runName': 'Run 3 - some model run', 'Training Info': 'Basic LR model for iris data', 'mlflow.log-model.history': '[{"run_id": "985ddfffee9745ad9d66d907ba179947", "artifact_path": "iris_model", "utc_time_created": "2025-05-19 18:04:37.243326", "model_uuid": "4da356cbd5c34a2391c937b6a426767f", "flavors": {"python_function": {"model_path": "model.pkl", "predict_fn": "predict", "loader_module": "mlflow.sklearn", "python_version": "3.12.10", "env": {"conda": "conda.yaml", "virtualenv": "python_env.

## Получить лучшую модель по метрике

In [52]:
experiment_name = "MLflow MLAdv practice"

# Получаем ID эксперимента по имени
# Получение клиента
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id

# Ищем запуски, сортируя по метрике "Logistic f1 score" по убыванию
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="",  # можно добавить фильтр, если нужно
    order_by=["metrics.`Logistic f1 score` DESC"],
    max_results=1
)

# Получение лучшего запуска
best_run = runs.iloc[0]
print("Best Run ID:", best_run.run_id)
print("F1 Score:", best_run["metrics.Logistic f1 score"])
print("Params:", {col.replace("params.", ""): best_run[col] for col in runs.columns if col.startswith("params.")})
print("Artifacts URI:", best_run.artifact_uri)

# Загрузка модели, если она была сохранена как "iris_model"
logged_model_uri = f"runs:/{best_run.run_id}/iris_model"
model = mlflow.sklearn.load_model(logged_model_uri)

Best Run ID: 985ddfffee9745ad9d66d907ba179947
F1 Score: 1.0
Params: {'random_state': '74', 'multi_class': 'multinomial', 'solver': 'lbfgs', 'max_iter': '200'}
Artifacts URI: mlflow-artifacts:/1/985ddfffee9745ad9d66d907ba179947/artifacts


In [53]:
model.predict(X_test)

array([1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2,
       0, 2, 2, 2, 2, 2, 0, 0])